# 05 - CPU worker: slot matching and per-pixel curve metrics

This notebook confirms `Predictor._cpu_worker`, the per-batch routine that (1) reshapes flat parameters, (2) sorts the ground-truth Gaussians by mean while pushing placeholder slots to the end, (3) reconstructs prediction and target profiles, and (4) computes per-pixel MSE / MAE / R-squared / cosine and the peak-index error. We hand-build parameters where the matched order and metric values are known in advance.

Modules exercised: `pipelines.inference_pipeline.predictor.Predictor._cpu_worker`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Construct a one-pixel batch with a known GT slot order

The worker expects flat parameter arrays of shape `(B, 3K, H, W)`. We use `B = 1`, `K = 2`, a single pixel, and choose the GT so its two Gaussians are given in **descending** mean order. The worker sorts GT by mean (ascending, placeholders last), so after matching the slot order must flip.

In [ ]:
from pipelines.inference_pipeline.predictor import Predictor

x_axis = np.linspace(-12.0, 12.0, 49).astype(np.float32)
K      = 2
B, Hp, Wp = 1, 1, 1

# GT given in descending mean order: slot0 mu=+5, slot1 mu=-5
gt_params = np.zeros((B, 3 * K, Hp, Wp), dtype=np.float32)
gt_params[0, 0:3, 0, 0] = [1.0, 5.0, 2.0]
gt_params[0, 3:6, 0, 0] = [0.8, -5.0, 2.0]

# Prediction: a perfect copy of GT (so the worker must still match by mean order)
pred_params = gt_params.copy()

print('GT slot means  (input order):', gt_params[0, 1, 0, 0], gt_params[0, 4, 0, 0])


## Run the worker and inspect the matched GT order

The worker's argument tuple is `(pred, gt, x_axis, n_gaussians, out_ch, norm_loc, norm_scale)`. The last three are unused by the numerical path here, so harmless placeholders are passed. The returned `gt_gauss_flat` is the GT after mean-sorting; its first slot must now be the `mu = -5` Gaussian.

In [ ]:
args = (pred_params, gt_params, x_axis, K, 3 * K,
        np.zeros(3 * K, np.float32), np.ones(3 * K, np.float32))

pred_curves, gt_curves, pred_flat, gt_flat, mets, peak = Predictor._cpu_worker(args)

print('matched GT slot0 mean:', gt_flat[0, 1, 0, 0], '(expected -5)')
print('matched GT slot1 mean:', gt_flat[0, 4, 0, 0], '(expected +5)')
print('pred_curves shape    :', pred_curves.shape, '(B, n_elev, H, W)')


## Perfect prediction yields trivial metrics

Since the prediction equals the GT, the per-pixel metrics must be the ideal values: MSE = 0, MAE = 0, R-squared = 1, cosine = 1, and peak-index error = 0.

In [ ]:
print('MSE      :', float(mets['mse'][0, 0, 0]))
print('MAE      :', float(mets['mae'][0, 0, 0]))
print('R^2      :', float(mets['r2'][0, 0, 0]))
print('cosine   :', float(mets['cos'][0, 0, 0]))
print('peak err :', float(peak[0, 0, 0]))


## A controlled imperfect prediction

Now we shift the prediction's first Gaussian mean by a few bins and rerun. The metrics must degrade in the expected direction (positive MSE/MAE, R-squared below 1, cosine below 1) and the predicted profile peak should move away from the target peak, producing a non-zero peak-index error.

In [ ]:
pred_shift = gt_params.copy()
pred_shift[0, 1, 0, 0] = 5.0 + 4.0   # shift the mu=+5 component to +9

args2 = (pred_shift, gt_params, x_axis, K, 3 * K,
         np.zeros(3 * K, np.float32), np.ones(3 * K, np.float32))
pc2, gc2, _, _, m2, peak2 = Predictor._cpu_worker(args2)

print('MSE      :', float(m2['mse'][0, 0, 0]))
print('R^2      :', float(m2['r2'][0, 0, 0]))
print('cosine   :', float(m2['cos'][0, 0, 0]))
print('peak err :', float(peak2[0, 0, 0]), 'elevation bins')


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.plot(x_axis, gc2[0, :, 0, 0], color='black', lw=1.8, label='GT (matched)')
ax.plot(x_axis, pc2[0, :, 0, 0], color='C3', lw=1.4, ls='--', label='Prediction (shifted)')
ax.fill_between(x_axis, pc2[0, :, 0, 0], gc2[0, :, 0, 0], color='C0', alpha=0.15)
ax.set_xlabel('elevation [m]')
ax.set_ylabel('intensity')
ax.set_title('CPU worker profiles: shifted prediction vs matched GT')
ax.legend()
fig.tight_layout()
plt.show()


## Expected visual outcome

The matched GT must list `mu = -5` before `mu = +5`, confirming the mean-sort. The perfect prediction must give MSE = 0, R-squared = 1, cosine = 1, peak error = 0. The shifted prediction must give a positive MSE, R-squared and cosine below one, and a peak error of several bins, with the plotted prediction curve visibly displaced from the GT around the right-hand peak.